# Portfolio Optimization & Efficient Frontier
### Markowitz framework applied to real market data

This notebook builds a **mean-variance optimized portfolio** from real equity data
downloaded via **yfinance**, implements the Markowitz efficient frontier through
Monte Carlo portfolio simulation and SciPy optimization, and computes key risk
metrics including **VaR**, **CVaR**, and **Sharpe ratio**.

**Key concepts covered:**
- Covariance matrix and pairwise asset correlations
- Efficient frontier (Monte Carlo random portfolios)
- Maximum Sharpe Ratio portfolio
- Minimum Variance portfolio
- Historical VaR and Conditional VaR (Expected Shortfall)
- Portfolio attribution: weights and contribution to risk

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import yfinance as yf
from scipy.optimize import minimize
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.titlesize": 13})

## 1. Universe Selection & Data Download

We select a diversified portfolio of **8 liquid equities** across sectors
(Technology, Finance, Healthcare, Energy, Consumer) to illustrate diversification benefits.

In [ ]:
TICKERS = {
    "AAPL":  "Apple (Tech)",
    "MSFT":  "Microsoft (Tech)",
    "JPM":   "JPMorgan (Finance)",
    "JNJ":   "Johnson & Johnson (Healthcare)",
    "XOM":   "ExxonMobil (Energy)",
    "LVMH.PA": "LVMH (Consumer/Luxury)",
    "TTE.PA":  "TotalEnergies (Energy)",
    "AIR.PA":  "Airbus (Industrials)",
}

START = "2020-01-01"
END   = "2024-12-31"

raw = yf.download(list(TICKERS.keys()), start=START, end=END,
                  auto_adjust=True, progress=False)["Close"]
prices = raw.dropna(how="all").ffill().dropna(how="any")

print(f"Period  : {START} → {END}")
print(f"Tickers : {list(prices.columns)}")
print(f"Rows    : {len(prices)} trading days")
prices.tail(3)

## 2. Returns & Descriptive Statistics

In [ ]:
returns = prices.pct_change().dropna()

annual_ret = returns.mean() * 252
annual_vol = returns.std() * np.sqrt(252)
sharpe     = annual_ret / annual_vol

stats = pd.DataFrame({
    "Ann. Return (%)":   (annual_ret * 100).round(2),
    "Ann. Volatility (%)": (annual_vol * 100).round(2),
    "Sharpe Ratio":      sharpe.round(2),
}).sort_values("Sharpe Ratio", ascending=False)

print(stats.to_string())

# Cumulative performance chart
cumul = (1 + returns).cumprod()
fig, ax = plt.subplots(figsize=(13, 5))
for col in cumul.columns:
    ax.plot(cumul.index, cumul[col], linewidth=1.5, label=col)
ax.set_title("Cumulative Performance (base 1 = 2020-01-01)")
ax.set_ylabel("Growth of $1")
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

## 3. Correlation Matrix

A low pairwise correlation is the foundation of diversification — combining assets
whose returns don't move together reduces portfolio volatility for a given return level.

In [ ]:
corr = returns.corr()

fig, ax = plt.subplots(figsize=(9, 7))
cax = ax.matshow(corr, cmap="RdYlGn", vmin=-1, vmax=1)
fig.colorbar(cax)
tickers_short = list(corr.columns)
ax.set_xticks(range(len(tickers_short)))
ax.set_yticks(range(len(tickers_short)))
ax.set_xticklabels(tickers_short, rotation=45, ha="left", fontsize=10)
ax.set_yticklabels(tickers_short, fontsize=10)
for i in range(len(tickers_short)):
    for j in range(len(tickers_short)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center",
                fontsize=8, color="black" if abs(corr.iloc[i, j]) < 0.7 else "white")
ax.set_title("Pairwise Correlation Matrix", pad=20)
plt.tight_layout()
plt.show()

## 4. Efficient Frontier via Monte Carlo Simulation

We randomly sample **20 000 portfolio weight vectors** and plot the resulting
(volatility, return) cloud. The upper-left boundary is the efficient frontier.

In [ ]:
np.random.seed(42)
N_PORT = 20_000
n      = len(prices.columns)

ann_ret = returns.mean().values * 252
ann_cov = returns.cov().values  * 252

port_returns = np.empty(N_PORT)
port_vols    = np.empty(N_PORT)
port_sharpes = np.empty(N_PORT)

r_f = 0.045

for i in range(N_PORT):
    w = np.random.dirichlet(np.ones(n))
    ret = w @ ann_ret
    vol = np.sqrt(w.T @ ann_cov @ w)
    port_returns[i] = ret
    port_vols[i]    = vol
    port_sharpes[i] = (ret - r_f) / vol

fig, ax = plt.subplots(figsize=(11, 6))
sc = ax.scatter(port_vols * 100, port_returns * 100,
                c=port_sharpes, cmap="plasma", s=3, alpha=0.5)
plt.colorbar(sc, ax=ax, label="Sharpe Ratio")
ax.set_title(f"Efficient Frontier — {N_PORT:,} Random Portfolios")
ax.set_xlabel("Annualised Volatility (%)")
ax.set_ylabel("Annualised Return (%)")
plt.tight_layout()
plt.show()

## 5. Optimal Portfolios via Numerical Optimisation

We use `scipy.optimize.minimize` (SLSQP) to find:
1. **Maximum Sharpe Ratio** portfolio (best risk-adjusted return)
2. **Minimum Variance** portfolio (lowest achievable volatility)

In [ ]:
def port_stats(w):
    ret = w @ ann_ret
    vol = np.sqrt(w.T @ ann_cov @ w)
    return ret, vol, (ret - r_f) / vol

constraints = [{"type": "eq", "fun": lambda w: w.sum() - 1}]
bounds = [(0.0, 1.0)] * n
w0 = np.ones(n) / n

res_sharpe = minimize(lambda w: -(w @ ann_ret - r_f) / np.sqrt(w.T @ ann_cov @ w),
                      w0, method="SLSQP", bounds=bounds, constraints=constraints)
res_minvar = minimize(lambda w: np.sqrt(w.T @ ann_cov @ w),
                      w0, method="SLSQP", bounds=bounds, constraints=constraints)

ret_sh, vol_sh, sr_sh = port_stats(res_sharpe.x)
ret_mv, vol_mv, sr_mv = port_stats(res_minvar.x)

# Replot frontier with optimal portfolios highlighted
fig, ax = plt.subplots(figsize=(11, 6))
sc = ax.scatter(port_vols * 100, port_returns * 100,
                c=port_sharpes, cmap="plasma", s=3, alpha=0.4)
plt.colorbar(sc, ax=ax, label="Sharpe Ratio")
ax.scatter(vol_sh * 100, ret_sh * 100, marker="*", s=400,
           color="gold", edgecolors="black", zorder=5, label=f"Max Sharpe (SR={sr_sh:.2f})")
ax.scatter(vol_mv * 100, ret_mv * 100, marker="D", s=200,
           color="cyan",  edgecolors="black", zorder=5, label=f"Min Variance (SR={sr_mv:.2f})")
ax.set_title("Efficient Frontier — Optimal Portfolios")
ax.set_xlabel("Annualised Volatility (%)")
ax.set_ylabel("Annualised Return (%)")
ax.legend()
plt.tight_layout()
plt.show()

# Weight comparison
tickers_list = list(prices.columns)
df_weights = pd.DataFrame({
    "Max Sharpe": res_sharpe.x * 100,
    "Min Variance": res_minvar.x * 100,
}, index=tickers_list).round(1)
print("\nPortfolio Weights (%)")
print(df_weights.to_string())

## 6. Weight Allocation — Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col, color in zip(axes, ["Max Sharpe", "Min Variance"], ["steelblue", "coral"]):
    vals = df_weights[col].sort_values(ascending=True)
    ax.barh(vals.index, vals.values, color=color, edgecolor="white")
    ax.set_title(f"{col} Portfolio")
    ax.set_xlabel("Weight (%)")
    ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

## 7. Historical VaR & CVaR for the Max-Sharpe Portfolio

In [ ]:
w_opt  = res_sharpe.x
port_r = returns @ w_opt  # daily portfolio returns

CONF_LEVELS = [0.95, 0.99]
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(port_r * 100, bins=80, color="steelblue", edgecolor="white",
        alpha=0.8, label="Daily returns")

for cl in CONF_LEVELS:
    var  = -np.percentile(port_r, (1 - cl) * 100)
    cvar = -port_r[port_r < -var].mean()
    ax.axvline(-var * 100, linestyle="--", linewidth=1.8,
               label=f"VaR {cl:.0%} = {var:.2%}  |  CVaR = {cvar:.2%}")

ax.set_title("Max-Sharpe Portfolio — Daily Return Distribution")
ax.set_xlabel("Daily Return (%)")
ax.legend()
plt.tight_layout()
plt.show()

print("\nRisk Metrics — Max Sharpe Portfolio (historical, 1-day horizon)")
for cl in CONF_LEVELS:
    var  = -np.percentile(port_r, (1 - cl) * 100)
    cvar = -port_r[port_r < -var].mean()
    print(f"  VaR  {cl:.0%}: {var:.2%}   CVaR {cl:.0%}: {cvar:.2%}")